# RAGAS 评测一个 RAG 系统（010 · 忠实度 / 相关性 / 检索层）

> 与章节 [07-RAG评测-RAGAS](../../08-RAG体系/07-RAG评测-RAGAS.md) 配套；语料与检索器沿用 [009-中文RAG全管线](009-中文RAG全管线.ipynb)。
> 一句话：**「能检索」不等于「系统能用」——这套 notebook 用 RAGAS 式三层指标，把「检索层顶格但生成层崩盘」照出来。**

本 notebook 的三层指标（全部免 LLM、CPU 可跑、数字可复现）：

| 层 | 指标 | 实现 | 解释 |
|---|---|---|---|
| 检索层 | Hit@5 Recall | BM25 / Dense 各自 top-5 是否含 gold | 找得到吗 |
| 生成层 | Faithfulness（4-gram 证据覆盖率） | 答案的 4-gram 有多少出现在喂入的上下文中 | 答得有所凭吗 |
| 生成层 | Answer Relevance（bge 余弦） | query 与答案的 bge-small-zh 余弦 | 答得贴题吗 |

然后做**判别性对照**：把答案换成「错误来源」，看指标是否显著下跌——评测指标最怕「怎么改都 0.9」。


## 1. 安装依赖（Colab 已含大部分）

In [1]:
import importlib, subprocess, sys
for pkg in ("rank_bm25", "jieba"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import faiss
except ImportError:
    for pkg in ("faiss", "faiss-cpu"):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            import faiss
            break
        except Exception:
            continue
print("deps ok")

deps ok


## 2. 数据：沿用 009 的 100 条中文百科语料

每条 `[上下文, gold问题]`，第 i 条的答案就在第 i 个上下文里（`gold_idx = i`），无需下载。

In [2]:
# 100 条中文百科上下文 + 每条一个 gold 问题（NOTEBOOK 内嵌，可直接跑）
CORPUS = [
 [
  "引力波是时空弯曲的涟漪，由大质量天体加速运动时产生，2015年首次被LIGO探测到",
  "引力波是2015年被哪个实验首次探测到的"
 ],
 [
  "量子纠缠指两个粒子无论相隔多远，其量子态都会相互关联，是量子通信的基础",
  "量子纠缠的特性和在通信中的作用是什么"
 ],
 [
  "Transformer是2017年提出的基于注意力机制的网络结构，是大语言模型的基础",
  "Transformer的核心机制是什么"
 ],
 [
  "梯度下降通过沿损失函数负梯度方向迭代更新参数，是训练神经网络的最基本优化算法",
  "训练神经网络最基本的优化算法是什么"
 ],
 [
  "图灵测试由阿兰·图灵在1950年提出，用于判断机器是否具备人类智能",
  "图灵测试是在哪一年由谁提出的"
 ],
 [
  "卷积神经网络通过卷积核在局部区域提取特征，特别适合图像识别任务",
  "卷积神经网络在什么任务上表现出色"
 ],
 [
  "反向传播算法通过链式法则逐层计算梯度，是深度学习训练的核心机制",
  "链式法则在深度学习训练中如何被应用"
 ],
 [
  "元学习的目标是让模型学会学习，在少量样本上快速适应新任务",
  "让模型从少量样本快速适应新任务的方法是什么"
 ],
 [
  "注意力机制允许模型在做预测时动态聚焦输入序列中最相关的部分",
  "注意力机制的核心思想是什么"
 ],
 [
  "大规模预训练语言模型通过在海量文本上预测下一个词来学习语言规律",
  "语言模型通过什么任务来学习语言规律"
 ],
 [
  "强化学习通过奖励信号引导智能体在环境中探索并优化策略",
  "强化学习中奖励信号的作用是什么"
 ],
 [
  "决策树通过递归划分特征空间来对数据进行分类或回归",
  "决策树的工作原理是什么"
 ],
 [
  "支持向量机通过寻找最大间隔超平面来分割不同类别的样本",
  "支持向量机的核心思路是什么"
 ],
 [
  "贝叶斯定理描述了在已知先验概率和观测数据后更新信念的方法",
  "贝叶斯定理主要用于做什么"
 ],
 [
  "过拟合是指模型在训练集上表现好但在新数据上表现差的现象",
  "什么是过拟合"
 ],
 [
  "K均值聚类通过迭代把样本划分为K个簇，使簇内距离最小",
  "K均值聚类的目标是什么"
 ],
 [
  "主成分分析通过线性变换把高维数据投影到低维，保留最大方差",
  "主成分分析的作用是什么"
 ],
 [
  "正则化通过在损失函数中加入模型复杂度惩罚项来抑制过拟合",
  "正则化的作用是什么"
 ],
 [
  "学习率决定了参数更新的步长，过大导致震荡过小导致收敛慢",
  "学习率过大会导致什么问题"
 ],
 [
  "批归一化通过在每层对激活值做归一化，加速深度学习训练收敛",
  "批归一化在训练中起什么作用"
 ],
 [
  "残差网络通过跳连接缓解深层网络训练中的梯度消失问题",
  "残差网络如何解决深层训练中的问题"
 ],
 [
  "长短期记忆网络通过门控机制解决传统RNN的长期依赖问题",
  "LSTM解决的核心问题是什么"
 ],
 [
  "知识蒸馏通过让学生模型模仿教师模型的软输出，把大模型能力压缩到小模型",
  "知识蒸馏的目标是什么"
 ],
 [
  "剪枝通过移除神经网络中不重要的权重或神经元来压缩模型",
  "神经网络剪枝的做法是什么"
 ],
 [
  "量化通过把模型权重从浮点数转换到更低比特表示来减小模型体积",
  "模型量化的目的是什么"
 ],
 [
  "联邦学习让多个设备在本地训练模型，只上传梯度而不共享原始数据",
  "联邦学习的核心特点是只上传什么"
 ],
 [
  "对比学习通过拉近正样本对、推远负样本对来学习良好表示",
  "对比学习的训练目标是什么"
 ],
 [
  "自编码器通过无监督方式学习数据的压缩表示并重构输入",
  "自编码器的用途是什么"
 ],
 [
  "生成对抗网络由生成器和判别器博弈训练，可生成逼真图像",
  "生成对抗网络包含哪两个部分"
 ],
 [
  "扩散模型通过逐步去噪从随机噪声生成数据，是图像生成的主流方法",
  "扩散模型的生成过程是怎样的"
 ],
 [
  "词向量用低维稠密向量表示词义，可通过余弦相似度衡量词间语义相近程度",
  "如何衡量两个词的语义相近程度"
 ],
 [
  "N-gram语言模型基于马尔可夫假设，假设当前词只依赖前n-1个词",
  "N-gram语言模型的马尔可夫假设是什么"
 ],
 [
  "BPE分词算法通过合并最频繁的字符对来构建词表，能有效处理未登录词",
  "BPE分词算法的核心操作是什么"
 ],
 [
  "词嵌入通过训练把词映射到向量空间，使语义相近的词距离更近",
  "词嵌入的目标是什么"
 ],
 [
  "提示工程通过精心设计指令引导大模型输出理想结果，是应用大模型的核心技能",
  "提示工程的主要意义是什么"
 ],
 [
  "思维链提示让模型分步骤推理问题，能显著提高复杂推理任务的表现",
  "思维链提示如何提高模型的推理能力"
 ],
 [
  "检索增强生成通过外部知识库检索相关文档来增强模型回答的准确性",
  "检索增强生成解决什么问题"
 ],
 [
  "向量数据库存储并检索高维向量，支持近似最近邻搜索，是RAG应用的基础设施",
  "向量数据库支持哪种搜索"
 ],
 [
  "限流算法通过令牌桶或滑动窗口控制系统在单位时间内处理的请求数",
  "常用的限流算法有哪些"
 ],
 [
  "数据库索引以B+树或哈希等结构加速数据查询，是数据库性能优化的核心",
  "数据库索引的主要作用是什么"
 ],
 [
  "缓存通过存储热点数据避免重复计算，从而大幅降低系统延迟",
  "缓存为何能降低系统延迟"
 ],
 [
  "负载均衡把请求分发到多个服务器，防止单点过载并提升可用性",
  "负载均衡的作用是什么"
 ],
 [
  "消息队列通过异步解耦组件，在流量高峰时削峰填谷，保证系统稳定",
  "消息队列在流量高峰时如何起作用"
 ],
 [
  "容器通过操作系统级虚拟化隔离应用，使部署快速且环境一致",
  "容器的核心优势是什么"
 ],
 [
  "分布式系统通过多台机器协同计算，在单个节点故障时仍能继续工作",
  "分布式系统的目标之一是容错具体是指什么"
 ],
 [
  "数据库事务保证一组操作要么全部成功要么全部回滚，具备原子性",
  "数据库事务的原子性是什么意思"
 ],
 [
  "布隆过滤器用位数组表示集合成员，占用极少的空间但允许小概率误判",
  "布隆过滤器在空间和准确性上有什么特点"
 ],
 [
  "一致性哈希在节点加入或退出时只影响少量数据迁移，是分布式缓存常用方案",
  "一致性哈希的优点是什么"
 ],
 [
  "断点续传通过记录已上传分片位置，在网络中断后能从中断处继续传输",
  "断点续传的原理是什么"
 ],
 [
  "压缩算法通过消除数据冗余降低存储和传输成本，是系统优化常用手段",
  "压缩算法降低的成本有哪些"
 ],
 [
  "视线跟踪结合眼动和头部姿态估计注视点，被用于用户交互和注意力分析",
  "人机交互中视线跟踪的应用是什么"
 ],
 [
  "中医学以阴阳五行理论为基础，通过辨证论治指导临床实践",
  "中医辨证论治在临床实践中的作用是什么"
 ],
 [
  "量子计算利用量子叠加和纠缠特性，在特定问题上可远超经典计算机",
  "量子计算为什么能超越经典计算机"
 ],
 [
  "黑洞是引力极强连光也无法逃逸的天体，其边界称为事件视界",
  "黑洞连什么也无法逃逸"
 ],
 [
  "光合作用是植物利用光能合成有机物的过程，是生态系统能量流动的起点",
  "光合作用在生态系统中扮演什么角色"
 ],
 [
  "人类基因组计划旨在确定人类基因组的全部DNA序列，是生命科学的基础工程",
  "人类基因组计划的目标是什么"
 ],
 [
  "疫苗通过刺激免疫系统产生记忆性应答，可在病原入侵时快速防御",
  "疫苗预防疾病的原理是什么"
 ],
 [
  "全球变暖主要由温室气体排放导致，会引发海平面上升和极端天气增多",
  "全球变暖的主要成因和影响有哪些"
 ],
 [
  "区块链通过哈希链和共识机制保证数据的不可篡改性，比特币是其著名的应用",
  "区块链保证数据不可篡改机制的叫什么"
 ],
 [
  "5G网络具备低时延高速率大连接的特性，是工业互联网和自动驾驶的关键",
  "5G网络的重要特性有哪些"
 ],
 [
  "自动驾驶通过传感器融合和决策规划，在复杂交通环境中安全驾行",
  "自动驾驶系统依赖哪两类核心技术"
 ],
 [
  "脑机接口通过采集脑电信号并经解码控制外部设备，可用于瘫痪患者康复",
  "脑机接口的应用场景有哪些"
 ],
 [
  "基因编辑技术CRISPR能精准修改特定基因位点，在疾病治疗上潜力巨大",
  "CRISPR技术的核心能力是什么"
 ],
 [
  "人造太阳托卡马克装置通过磁场约束等离子体，探索可控核聚变能源",
  "托卡马克装置的目标是什么"
 ],
 [
  "超级计算机用加快气象预报药物研发等科学计算，其算力通常用浮点运算次数衡量",
  "超级计算机的算力通常用什么衡量"
 ],
 [
  "湿地被称为地球之肾，在净化水质调蓄洪水维护生物多样性方面发挥关键作用",
  "湿地为什么被称为地球之肾"
 ],
 [
  "青藏高原被称为亚洲水塔，是长江黄河等大河的源头",
  "亚洲水塔指的是哪里"
 ],
 [
  "大熊猫的食性已特化为以竹子为主，消化系统仍保留肉食动物的特征",
  "大熊猫的食性特点是什么"
 ],
 [
  "福建土楼多为客家人所建，以厚墙圆形或方形布局在防御和宗族聚居上独具特色",
  "福建土楼的建造者和特点是什么"
 ],
 [
  "都江堰是李冰父子主持修建的水利工程，引水灌溉成都平原两千多年",
  "都江堰水利工程是谁主持修建的"
 ],
 [
  "清明上河图描绘北宋都城汴京的市井生活，是研究宋代社会的重要史料",
  "清明上河图描绘的是哪个朝代的都城"
 ],
 [
  "彗星主要由冰和尘埃组成，接近太阳时会形成长长的彗尾",
  "彗星主要由什么组成"
 ],
 [
  "极光是太阳风带电粒子撞击高层大气分子产生的发光现象",
  "极光是如何产生的"
 ],
 [
  "海啸通常由海底地震或火山喷发引发，目前只能预警难以完全防御",
  "海啸通常由什么引发"
 ],
 [
  "沙漠化是土地因过度放牧开垦等原因退化，严重威胁粮食安全",
  "沙漠化的主要原因有哪些"
 ],
 [
  "碳中和指通过减排和碳汇使二氧化碳排放量达到平衡，是全球气候目标",
  "碳中和的目标是什么"
 ],
 [
  "芯片制程越小晶体管密度越高功耗越低，是半导体产业竞争的核心指标",
  "芯片制程缩小的意义是什么"
 ],
 [
  "熔断机制在股票指数跌至阈值时暂停交易，防止市场恐慌式下跌",
  "股市熔断机制的设置目的是什么"
 ],
 [
  "量化交易通过数学模型和程序化下单捕捉市场套利与回撤机会",
  "量化交易的特点是什么"
 ],
 [
  "复利俗称利滚利，指利息自产生起再计入本金继续生息",
  "复利的通俗说法是什么"
 ],
 [
  "市盈率是股价与每股收益的比值，估值高低需结合行业成长性来看",
  "市盈率是哪个指标与每股收益的比值"
 ],
 [
  "分散投资把资金配置到不同类型的资产以降低单一资产下跌的冲击",
  "投资组合分散投资的目的是什么"
 ],
 [
  "通货膨胀指货币购买力下降物价总水平持续上升，美联储通过加息收水控制通胀",
  "应对通胀通常采用的货币政策工具是什么"
 ],
 [
  "碳交易市场给二氧化碳排放定价，让减产排放的企业能出售配额获利",
  "碳交易市场起什么作用"
 ],
 [
  "供应链安全指关键原材料和零部件的供应稳定，是制造业的核心关切",
  "供应链安全主要指什么"
 ],
 [
  "反应堆堆芯需持续冷却，一旦失去冷却将可能导致堆芯熔毁事故",
  "核电站反应堆失去冷却可能导致的后果是什么"
 ],
 [
  "空间站需要氧气水等生命保障系统为航天员长时间驻留创造条件",
  "空间站的生命保障系统为航天员提供什么"
 ],
 [
  "探索合成生物学通过改造基因回路让微生物生产药物燃料等物质",
  "合成生物学的应用方向有哪些"
 ],
 [
  "柔性电子使设备可弯曲折叠，是未来可穿戴设备的重要方向",
  "柔性电子的主要优势是什么"
 ],
 [
  "星链通过低轨卫星组网为地面提供高速网络，特点是延迟低覆盖广",
  "星链的网络特征是什么"
 ],
 [
  "长征火箭是中国进入太空的主要运载工具，探索月球和空间站建设依赖它",
  "长征火箭的作用是什么"
 ],
 [
  "前庭觉负责感知头部的倾斜和旋转，是平衡系统的关键",
  "前庭觉在人体中起什么作用"
 ],
 [
  "激素由内分泌腺分泌，随血液循环到达靶器官调节生理活动",
  "激素是如何运输到靶器官的"
 ],
 [
  "免疫系统的记忆细胞能在二次感染时快速产生更强的免疫应答",
  "免疫记忆细胞在二次感染时有何表现"
 ],
 [
  "人工智能对齐指让模型的价值观与人类意图一致，是安全部署大模型的关键",
  "什么是人工智能对齐"
 ],
 [
  "幻觉是生成式模型输出与事实不符内容的现象，需要通过检索验证等手段缓解",
  "生成式模型输出与事实不符的现象叫什么"
 ],
 [
  "提示注入是通过恶意指令让大模型违反设定，是LLM应用的主要安全威胁",
  "提示注入攻击的原理是什么"
 ],
 [
  "联邦蒸馏结合联邦学习与知识蒸馏，在保护隐私的同时聚合各端模型能力",
  "联邦蒸馏结合了哪两种技术"
 ],
 [
  "边缘计算把计算任务下沉到靠近数据源的设备，降低时延减少带宽消耗",
  "边缘计算降低的是什么"
 ],
 [
  "RDMA允许数据绕过操作系统内核直达网卡内存，显著降低网络延迟",
  "RDMA降低网络延迟的原理是什么"
 ]
]
corpus_q = [c for c, _ in CORPUS]     # 检索语料 = 全部上下文
test_q = [q for _, q in CORPUS]       # 测试问题 = 全部 gold 问题
gold_idx = list(range(len(corpus_q))) # 第 i 条问题的答案就在第 i 个上下文里
print("语料条目:", len(corpus_q), " 测试问题:", len(test_q))

语料条目: 100  测试问题: 100


## 3. 检索器：BM25 + bge 稠密（含探针）

沿用 009 的检索栈；先用 Hit@5 把「检索层有没有张力」探明白。


In [3]:
import jieba, faiss, numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def cut(s):
    return [w for w in jieba.cut(s) if w.strip() and len(w.strip()) > 1]

# -- 稀疏：BM25（中文先 jieba 分词） --
bm = BM25Okapi([cut(c) for c in corpus_q])

# -- 稠密：bge-small-zh-v1.5（约 95MB，CPU 可跑） + faiss 内积索引 --
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")
corpus_emb = model.encode(corpus_q, normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(np.ascontiguousarray(corpus_emb))

def search_bm25(q, n=5):
    return np.argsort(bm.get_scores(cut(q)))[::-1][:n].tolist()

def search_dense(q, n=5):
    v = model.encode([q], normalize_embeddings=True)
    _, idx = index.search(np.ascontiguousarray(v), n)
    return idx[0].tolist()

# 检索层探针：两条路 Hit@5 是否顶格
print("检索层 Hit@5: BM25=%.4f  Dense=%.4f" % (
    np.mean([1.0 if i in search_bm25(q, 5) else 0.0 for i, q in enumerate(test_q)]),
    np.mean([1.0 if i in search_dense(q, 5) else 0.0 for i, q in enumerate(test_q)])))

检索层 Hit@5: BM25=1.0000  Dense=1.0000


## 4. RAGAS 式指标实现（免 LLM，4-gram / 余弦）

与章节 07 第 5 节的 probe 同款实现，保证文中的数字可在本 notebook 复现：
- `faithfulness_proxy`：答案 4-gram 在「喂入上下文（top-3 拼接）」中的出现比例 → 证据覆盖率。
- `answer_relevance`：`query` 与 `答案` 的 bge 余弦 → 贴题度。


In [4]:
def ngrams4(s):
    return {s[i:i+4] for i in range(len(s)-3)}

def faithfulness_proxy(ans, ctx):
    g = ngrams4(ans); m = len(g) or 1
    return len({x for x in g if x in ctx}) / m

def answer_relevance(q, ans):
    qv = model.encode([q], normalize_embeddings=True)[0]
    av = model.encode([ans], normalize_embeddings=True)[0]
    return float(qv @ av)

print("指标就绪：faithfulness_proxy / answer_relevance")

指标就绪：faithfulness_proxy / answer_relevance


## 5. 实验 A：正确来源基线（抽取式答案，20 样本）

`RandomState(0)` 固定抽 20 条（与 009 同题面可比）。答案 = 检索 top-1 上下文（抽取式基线，便于免模型复现）。

In [5]:
rng = np.random.RandomState(0)
SAMPLE = rng.choice(np.arange(len(corpus_q)), size=20, replace=False).tolist()

hits, faiths, rels = [], [], []
for i in SAMPLE:
    q = test_q[i]
    top = search_dense(q, 3)                       # RAG: 取 top-3 上下文
    ans = corpus_q[top[0]]                         # 答案 = top-1 上下文（抽取式）
    ctx = " ".join(corpus_q[j] for j in top)       # 喂进"大模型"的上下文
    hits.append(1.0 if i in search_dense(q, 5) else 0.0)
    faiths.append(faithfulness_proxy(ans, ctx))
    rels.append(answer_relevance(q, ans))

print("样本数: %d (USE_LLM=0 抽取式基线, top-3 上下文)" % len(SAMPLE))
print("检索层  Hit@5 Recall                     : %.4f" % np.mean(hits))
print("生成层  Faithfulness(4-gram证据覆盖率)     : %.4f" % np.mean(faiths))
print("生成层  Answer Relevance(bge余弦)        : %.4f (±%.3f)" % (np.mean(rels), np.std(rels)))

样本数: 20 (USE_LLM=0 抽取式基线, top-3 上下文)
检索层  Hit@5 Recall                     : 1.0000
生成层  Faithfulness(4-gram证据覆盖率)     : 1.0000
生成层  Answer Relevance(bge余弦)        : 0.7362 (±0.069)


## 6. 实验 B：错误来源对照（判别性检验）

把答案**故意换成错误来源**（检索 top-1 之后的第 3 条，`(top[0]+3) % N`），其余不动。
好指标必须能抓住这种「坏答案」——两张表都改不出来变动的评测台是废的。

In [6]:
bad_faith, bad_rel = [], []
for i in SAMPLE[:6]:
    q = test_q[i]
    top = search_dense(q, 3)
    ans = corpus_q[(top[0] + 3) % len(corpus_q)]      # 故意错来源
    ctx = " ".join(corpus_q[j] for j in top)
    bad_faith.append(faithfulness_proxy(ans, ctx))
    bad_rel.append(answer_relevance(q, ans))
print("对照-错来源答案  →  Faithfulness %.4f / Relevance %.4f（应显著下降）"
      % (np.mean(bad_faith), np.mean(bad_rel)))

对照-错来源答案  →  Faithfulness 0.1667 / Relevance 0.4018（应显著下降）


## 7. 汇总对比 + 落盘

逐条（前 6 样本）看正确 vs 错误来源的四列，再给全样本聚合。

In [7]:
import pathlib, pandas as pd
pathlib.Path("out").mkdir(exist_ok=True)

# 全样本聚合（实验 A 复用上文算出的向量，避免重复 embedding）
A_f, A_r = float(np.mean(faiths)), float(np.mean(rels))
B_f, B_r = float(np.mean(bad_faith)), float(np.mean(bad_rel))
print("管线A(正确来源)  Faithfulness=%.4f  Relevance=%.4f" % (A_f, A_r))
print("管线B(错误来源)  Faithfulness=%.4f  Relevance=%.4f" % (B_f, B_r))
print("判别性: F %.2f→%.2f, R %.2f→%.2f  —— 指标能抓住坏答案" % (A_f, B_f, A_r, B_r))

rows = []
for i in SAMPLE[:6]:
    q = test_q[i]
    top = search_dense(q, 3)
    good = corpus_q[top[0]]
    bad = corpus_q[(top[0] + 3) % len(corpus_q)]
    ctx = " ".join(corpus_q[j] for j in top)
    rows.append([q[:16], faithfulness_proxy(good, ctx), answer_relevance(q, good),
                 faithfulness_proxy(bad, ctx), answer_relevance(q, bad)])
df = pd.DataFrame(rows, columns=["查询(截)", "F·正确来源", "R·正确来源", "F·错误来源", "R·错误来源"])
df.to_csv("out/010_ragas_metrics.csv", index=False, encoding="utf-8-sig")
print("\n", df.round(3).to_markdown(index=False), sep="")

管线A(正确来源)  Faithfulness=1.0000  Relevance=0.7362
管线B(错误来源)  Faithfulness=0.1667  Relevance=0.4018
判别性: F 1.00→0.17, R 0.74→0.40  —— 指标能抓住坏答案

| 查询(截)                         |   F·正确来源 |   R·正确来源 |   F·错误来源 |   R·错误来源 |
|:---------------------------------|-------------:|-------------:|-------------:|-------------:|
| 对比学习的训练目标是什么         |            1 |        0.619 |            0 |        0.37  |
| 空间站的生命保障系统为航天员提供 |            1 |        0.844 |            1 |        0.517 |
| Transformer的核心机制            |            1 |        0.714 |            0 |        0.499 |
| 人类基因组计划的目标是什么       |            1 |        0.798 |            0 |        0.328 |
| 碳中和的目标是什么               |            1 |        0.765 |            0 |        0.293 |
| 免疫记忆细胞在二次感染时有何表现 |            1 |        0.89  |            0 |        0.403 |
